In [ ]:
import os
import torch
from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

print("Loading models...")

# --------------------------
# Memory optimization for 4GB RAM
# --------------------------
torch.set_num_threads(2)

# --------------------------
# Embedding model for RAG
# --------------------------
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# --------------------------
# Generator model (FLAN-T5-small - actually works for Q&A)
# --------------------------
model_name = "google/flan-t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Models loaded!")

# --------------------------
# Load HR documents
# --------------------------
docs_folder = "../hr_docs"
knowledge_base = []

# Check if folder exists
if not os.path.exists(docs_folder):
    print(f"Creating folder: {docs_folder}")
    os.makedirs(docs_folder)
    
    # Create sample HR documents
    sample_docs = {
        "attendance.txt": """Attendance Policy
Employees must mark their attendance daily through the HR portal.
The attendance system can be accessed at: https://hr.company.com/attendance
You can also use the mobile app "HR Connect" to mark attendance.
Attendance should be marked before 10:00 AM each day.""",

        "leave_policy.txt": """Leave Application Process
To apply for leave, log into the HR portal and click on "Leave Requests".
Fill out the leave application form with dates and reason.
Submit at least 3 days in advance for planned leave.
For sick leave, notify your manager by 9:00 AM on the day of absence.""",

        "hr_policies.txt": """General HR Policies
Vacation: 20 days per year
Sick leave: 10 days per year
Remote work: Up to 3 days per week
Health insurance: After 3 months of employment"""
    }
    
    for filename, content in sample_docs.items():
        with open(os.path.join(docs_folder, filename), "w", encoding="utf-8") as f:
            f.write(content)
    print("Created sample HR documents")

# Load all documents
for file in os.listdir(docs_folder):
    if file.endswith(".txt"):
        with open(os.path.join(docs_folder, file), "r", encoding="utf-8") as f:
            text = f.read()
            # Split by double newlines for paragraphs
            chunks = text.split("\n\n")
            # Also split by single newlines if no double newlines
            if len(chunks) == 1:
                chunks = text.split("\n")
            # Filter out empty chunks
            chunks = [chunk.strip() for chunk in chunks if chunk.strip()]
            knowledge_base.extend(chunks)
            print(f"Loaded {len(chunks)} chunks from {file}")

print(f"\nTotal: {len(knowledge_base)} HR paragraphs loaded")

if len(knowledge_base) == 0:
    knowledge_base = [
        "Mark attendance through HR portal at https://hr.company.com/attendance",
        "Apply for leave through HR portal under 'Leave Requests'",
        "Vacation: 20 days per year. Sick leave: 10 days per year."
    ]
    print("Using default HR policies")

# Create embeddings
print("\nCreating embeddings...")
embeddings = embed_model.encode(knowledge_base, show_progress_bar=True)
print("Embeddings created!")

# --------------------------
# Search function
# --------------------------
def search_docs(query, top_k=3):
    query_emb = embed_model.encode([query])
    sims = cosine_similarity(query_emb, embeddings)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    return "\n".join([knowledge_base[i] for i in top_idx])

# --------------------------
# Generate answer using FLAN-T5
# --------------------------
def generate_answer(question):
    q = question.lower().strip()
    
    # Quick responses for greetings
    if q in ["hi", "hello", "hey", "hii", "helloo"]:
        return "Hello! I'm your HR assistant. How can I help you today?"
    
    # Get relevant context from documents
    context = search_docs(question)
    
    # Create prompt for FLAN-T5
    prompt = f"""Answer the following question based on the HR information provided.

Information:
{context}

Question: {question}

Answer:"""
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=50,
            num_beams=2,
            temperature=0.7,
            do_sample=True,
            early_stopping=True
        )
    
    # Decode the answer
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # If answer is empty or too short, provide fallback
    if not answer or len(answer) < 5:
        answer = "I found this information that might help: " + context[:200]
    
    return answer

# --------------------------
# Chat loop
# --------------------------
print("\n" + "="*50)
print("🤖 HR Assistant - Using FLAN-T5")
print("="*50)
print("\nAsk me about:")
print("• Attendance marking")
print("• Leave application")
print("• HR policies")
print("• Vacation days")
print("\nType 'exit' to quit\n")

while True:
    question = input("\n👤 You: ").strip()
    
    if question.lower() in ["exit", "quit", "bye"]:
        print("🤖 Bot: Goodbye! Have a great day!")
        break
    
    if not question:
        continue
    
    print("🤖 Bot: ", end="", flush=True)
    answer = generate_answer(question)
    print(answer)

Loading models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

🤖 Bot: I found this information that might help: Q: sure 
A: yeah sure but you also mail in administration@lsituoe.edu.pk for more information
Q: Are you there
A: I'm always here! 24/7 ready to help with your HR questions. What do you need?
Q: Hey
A



👤 You:  where i get mark leave 


🤖 Bot: you apply leave in leave page



👤 You:  sure 


🤖 Bot: I found this information that might help: Q: sure 
A: yeah sure but you also mail in administration@lsituoe.edu.pk for more information
Q: Are you there
A: I'm always here! 24/7 ready to help with your HR questions. What do you need?
Q: Hey
A
